# Part 11: MST Proofs and Diff

This notebook covers the read-side operations on MSTs: proof paths
that demonstrate a key belongs to the tree, the MST walker that
iterates all entries, and diff operations that compare two MST
versions to produce a list of changes.

**What you'll learn:**
- Proof path: the sequence of CIDs from root to a leaf
- MST walker: in-order traversal of all key-value pairs
- MST diff: comparing two trees to produce add/update/delete ops
- How diffs feed into the firehose and sync protocol

**Prerequisites:** Part 10 (MST Insertion)

**Estimated Time:** 20-25 minutes

## Step 1: Proof Path

A proof path demonstrates that a key exists in the MST. It consists of
the CIDs of each node along the path from the root to the node containing
the key. In ATProto, proof paths are used for repository verification.

In [ ]:
// Proof path — list of CIDs from root to target node
@interface MSTProofPath : NSObject
@property (nonatomic, strong) NSMutableArray *nodeCIDs;
- (void)addNodeCID:(NSString *)cid;
- (BOOL)verifyContainsKey:(NSString *)key inTree:(NSDictionary *)treeIndex;
@end

@implementation MSTProofPath
- (instancetype)init {
    self = [super init];
    if (self) { _nodeCIDs = [NSMutableArray array]; }
    return self;
}
- (void)addNodeCID:(NSString *)cid {
    [_nodeCIDs addObject:cid];
}
- (BOOL)verifyContainsKey:(NSString *)key inTree:(NSDictionary *)treeIndex {
    // treeIndex maps CID -> {keys: [...], subtrees: [...]}
    // Walk the proof path, checking each node
    for (int i = 0; i < [_nodeCIDs count]; i++) {
        NSString *nodeCID = [_nodeCIDs objectAtIndex:i];
        NSDictionary *node = [treeIndex objectForKey:nodeCID];
        if (node == nil) {
            NSLog(@"Missing node: %@", nodeCID);
            return NO;
        }
        NSArray *keys = [node objectForKey:@"keys"];
        for (int j = 0; j < [keys count]; j++) {
            if ([[keys objectAtIndex:j] isEqualToString:key]) {
                NSLog(@"Found '%@' in node %@", key, nodeCID);
                return YES;
            }
        }
    }
    NSLog(@"Key '%@' not found in proof path", key);
    return NO;
}
@end

// Build a simple tree index
NSDictionary *treeIdx = @{
    @"bafyreiroot": @{
        @"keys": @[@"app.bsky.feed.post/abc", @"app.bsky.feed.post/def"],
        @"subtrees": @[@"bafyreichild"]
    },
    @"bafyreichild": @{
        @"keys": @[@"app.bsky.actor.profile/self"],
        @"subtrees": @[]
    }
};

MSTProofPath *proof = [[MSTProofPath alloc] init];
[proof addNodeCID:@"bafyreiroot"];
[proof addNodeCID:@"bafyreichild"];

NSLog(@"Verify profile: %d", [proof verifyContainsKey:@"app.bsky.actor.profile/self" inTree:treeIdx]);
NSLog(@"Verify missing: %d", [proof verifyContainsKey:@"app.bsky.feed.post/xyz" inTree:treeIdx]);

## Step 2: MST Walker

The MST walker performs an in-order traversal of all key-value pairs.
This is used for repository export, CAR file generation, and listing
all records in a repository.

In [ ]:
// MST walker — in-order traversal
@interface MSTWalker : NSObject
- (NSMutableArray *)walkTreeIndex:(NSDictionary *)treeIndex rootCID:(NSString *)rootCID;
@end

@implementation MSTWalker
- (NSMutableArray *)walkTreeIndex:(NSDictionary *)treeIndex rootCID:(NSString *)rootCID {
    NSMutableArray *entries = [NSMutableArray array];
    [self walkNode:rootCID treeIndex:treeIndex entries:entries];
    return entries;
}
- (void)walkNode:(NSString *)cid treeIndex:(NSDictionary *)treeIndex entries:(NSMutableArray *)entries {
    NSDictionary *node = [treeIndex objectForKey:cid];
    if (node == nil) return;
    
    NSArray *keys = [node objectForKey:@"keys"];
    NSArray *subtrees = [node objectForKey:@"subtrees"];
    
    // Walk subtrees and keys in interleaved order
    // Simplified: walk all subtrees first, then all keys
    for (int i = 0; i < [subtrees count]; i++) {
        [self walkNode:[subtrees objectAtIndex:i] treeIndex:treeIndex entries:entries];
    }
    for (int i = 0; i < [keys count]; i++) {
        [entries addObject:[keys objectAtIndex:i]];
    }
}
@end

MSTWalker *walker = [[MSTWalker alloc] init];
NSMutableArray *allEntries = [walker walkTreeIndex:treeIdx rootCID:@"bafyreiroot"];
NSLog(@"All entries (%d):", [allEntries count]);
for (int i = 0; i < [allEntries count]; i++) {
    NSLog(@"  %@", [allEntries objectAtIndex:i]);
}

## Step 3: MST Diff

Comparing two MST versions produces a list of diff operations.
This is the same concept from Part 16 (Firehose & Sync), but here
we focus on how the MST structure itself produces the diff.

The diff walks both trees in parallel, comparing keys and CIDs.

In [ ]:
// MST diff — compare two tree versions
@interface MSTDiff : NSObject
- (NSMutableArray *)diffOldEntries:(NSArray *)oldEntries newEntries:(NSArray *)newEntries;
@end

@implementation MSTDiff
- (NSMutableArray *)diffOldEntries:(NSArray *)oldEntries newEntries:(NSArray *)newEntries {
    // Build lookup sets
    NSMutableDictionary *oldMap = [NSMutableDictionary dictionary];
    NSMutableDictionary *newMap = [NSMutableDictionary dictionary];
    
    for (int i = 0; i < [oldEntries count]; i++) {
        NSDictionary *e = [oldEntries objectAtIndex:i];
        [oldMap setObject:[e objectForKey:@"cid"] forKey:[e objectForKey:@"key"]];
    }
    for (int i = 0; i < [newEntries count]; i++) {
        NSDictionary *e = [newEntries objectAtIndex:i];
        [newMap setObject:[e objectForKey:@"cid"] forKey:[e objectForKey:@"key"]];
    }
    
    NSMutableArray *ops = [NSMutableArray array];
    
    // Find adds and updates
    for (NSString *key in newMap) {
        NSString *newCid = [newMap objectForKey:key];
        NSString *oldCid = [oldMap objectForKey:key];
        if (oldCid == nil) {
            [ops addObject:@{@"action": @"add", @"key": key, @"cid": newCid}];
        } else if (![newCid isEqualToString:oldCid]) {
            [ops addObject:@{@"action": @"update", @"key": key, @"prevCid": oldCid, @"cid": newCid}];
        }
    }
    
    // Find deletes
    for (NSString *key in oldMap) {
        if ([newMap objectForKey:key] == nil) {
            [ops addObject:@{@"action": @"delete", @"key": key, @"prevCid": [oldMap objectForKey:key]}];
        }
    }
    
    return ops;
}
@end

NSArray *old = @[
    @{@"key": @"app.bsky.feed.post/abc", @"cid": @"bafyrei111"},
    @{@"key": @"app.bsky.feed.post/def", @"cid": @"bafyrei222"},
    @{@"key": @"app.bsky.actor.profile/self", @"cid": @"bafyrei333"}
];

NSArray *new_ = @[
    @{@"key": @"app.bsky.feed.post/abc", @"cid": @"bafyrei111"},
    @{@"key": @"app.bsky.feed.post/def", @"cid": @"bafyrei999"},
    @{@"key": @"app.bsky.feed.post/xyz", @"cid": @"bafyrei444"}
];

MSTDiff *differ = [[MSTDiff alloc] init];
NSMutableArray *ops = [differ diffOldEntries:old newEntries:new_];
NSLog(@"Diff operations: %d", [ops count]);
for (int i = 0; i < [ops count]; i++) {
    NSDictionary *op = [ops objectAtIndex:i];
    NSLog(@"  %@ %@", [op objectForKey:@"action"], [op objectForKey:@"key"]);
}

## Step 4: Garazyk Production Anchor

In the Garazyk codebase:

- `MSTWalker` — walks the MST in key order, yielding entries
- `MST.diffFrom:` — compares two MST root CIDs, producing diff ops
- `RepoCommit` — wraps MST diffs into commit events for the firehose

The production walker uses CID-based traversal through the block store,
while our notebook uses an in-memory tree index for simplicity.

## Exercise

Add a `countEntries` method to `MSTWalker` that returns the total number
of key-value pairs in the tree without building the full entries array.
This is useful for checking repository size without materializing all entries.

Hint: instead of appending to an array, increment a counter.

In [ ]:
// Exercise: implement countEntries
// Your code here...

## Solution

In [ ]:
// Solution: countEntries returns total without building array
@interface MSTWalker2 : NSObject
- (int)countEntriesInTreeIndex:(NSDictionary *)treeIndex rootCID:(NSString *)rootCID;
@end

@implementation MSTWalker2
- (int)countEntriesInTreeIndex:(NSDictionary *)treeIndex rootCID:(NSString *)rootCID {
    return [self countNode:rootCID treeIndex:treeIndex];
}
- (int)countNode:(NSString *)cid treeIndex:(NSDictionary *)treeIndex {
    NSDictionary *node = [treeIndex objectForKey:cid];
    if (node == nil) return 0;
    
    NSArray *keys = [node objectForKey:@"keys"];
    NSArray *subtrees = [node objectForKey:@"subtrees"];
    
    int count = [keys count];
    for (int i = 0; i < [subtrees count]; i++) {
        count = count + [self countNode:[subtrees objectAtIndex:i] treeIndex:treeIndex];
    }
    return count;
}
@end

MSTWalker2 *w2 = [[MSTWalker2 alloc] init];
int total = [w2 countEntriesInTreeIndex:treeIdx rootCID:@"bafyreiroot"];
NSLog(@"Total entries: %d (expect 3)", total);

## What to Read Next

You now understand MST proofs and diffs. Next:
- **Part 12: XRPC Dispatch** — how method names route to handlers
- **Part 14: Record Writes** — how records are stored in the MST
- **Part 16: Firehose & Sync** — how diffs become firehose events